In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ==============================================
# STEP 1 — Structural / Deterministic Image Preprocessing
# IFND paper aligned (256×256 resize)
# Includes orientation fix (EXIF)
# ==============================================

import os
from PIL import Image, ImageOps
from tqdm import tqdm

# Paths
BASE = "/content/drive/MyDrive/major_project/IFND_dataset"
RAW_IMG_DIR = f"{BASE}/images"
CLEAN_IMG_DIR = f"{BASE}/images_clean"

os.makedirs(CLEAN_IMG_DIR, exist_ok=True)

# Target size (as per IFND paper)
TARGET_SIZE = (256, 256)

# Counters
stats = {
    "processed": 0,
    "skipped_corrupt": 0,
    "already_exists": 0
}

# List all raw images
image_files = [f for f in os.listdir(RAW_IMG_DIR) if f.lower().endswith(".jpg")]

print(f"🖼️ Total raw images found: {len(image_files)}")

for img_name in tqdm(image_files, desc="Cleaning images"):
    raw_path = os.path.join(RAW_IMG_DIR, img_name)
    clean_path = os.path.join(CLEAN_IMG_DIR, img_name)

    # Skip if already processed (resume-safe)
    if os.path.exists(clean_path):
        stats["already_exists"] += 1
        continue

    try:
        with Image.open(raw_path) as img:
            # 🔹 STEP 1: Orientation fix (EXIF-based)
            img = ImageOps.exif_transpose(img)

            # 🔹 STEP 2: Convert to RGB
            img = img.convert("RGB")

            # 🔹 STEP 3: Resize to 256×256 (paper aligned)
            img = img.resize(TARGET_SIZE)

            # 🔹 STEP 4: Save clean image
            img.save(clean_path, format="JPEG", quality=95)

            stats["processed"] += 1

    except Exception:
        # Corrupted or unreadable image
        stats["skipped_corrupt"] += 1
        continue

# Summary
print("\n✅ STEP-1 COMPLETED — Image Cleaning Summary:")
for k, v in stats.items():
    print(f"{k}: {v}")

print("\n📁 Clean images saved in:", CLEAN_IMG_DIR)


🖼️ Total raw images found: 38578


Cleaning images:  54%|█████▎    | 20649/38578 [1:14:00<1:20:03,  3.73it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Cleaning images: 100%|██████████| 38578/38578 [2:34:41<00:00,  4.16it/s]


✅ STEP-1 COMPLETED — Image Cleaning Summary:
processed: 34527
skipped_corrupt: 4051
already_exists: 0

📁 Clean images saved in: /content/drive/MyDrive/major_project/IFND_dataset/images_clean


In [ ]:
import os

BASE = "/content/drive/MyDrive/major_project/IFND_dataset"
CLEAN_IMG_DIR = f"{BASE}/images_clean"

clean_images = [f for f in os.listdir(CLEAN_IMG_DIR) if f.lower().endswith(".jpg")]

print("🖼️ Total clean images:", len(clean_images))

🖼️ Total clean images: 34527


In [ ]:
RAW_IMG_DIR = f"{BASE}/images"

raw_images = [f for f in os.listdir(RAW_IMG_DIR) if f.lower().endswith(".jpg")]
clean_images = [f for f in os.listdir(CLEAN_IMG_DIR) if f.lower().endswith(".jpg")]

print("Raw images count   :", len(raw_images))
print("Clean images count :", len(clean_images))
print("Removed / skipped  :", len(raw_images) - len(clean_images))



Raw images count   : 38578
Clean images count : 34527
Removed / skipped  : 4051


In [ ]:
# ==============================================
# STEP 2 — Merge text + image paths
# Create STRICT multimodal IFND_multimodel.csv
# ==============================================

import os
import pandas as pd

BASE = "/content/drive/MyDrive/major_project/IFND_dataset"
CLEAN_IMG_DIR = f"{BASE}/images_clean"

# load cleaned text CSV
df = pd.read_csv(f"{BASE}/finally_IFND_is_cleaned.csv")

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

# attach image path (relative)
def get_image_path(news_id):
    rel_path = f"images_clean/{news_id}.jpg"
    full_path = os.path.join(BASE, rel_path)
    return rel_path if os.path.exists(full_path) else None

df["image_path"] = df["id"].apply(get_image_path)

# 🔴 IMPORTANT: drop rows without image
df = df[df["image_path"].notna()].reset_index(drop=True)

# reorder columns
df = df[["id", "Clean_Text", "image_path", "Label"]]

# save strict multimodal CSV
out_path = f"{BASE}/IFND_multimodel.csv"
df.to_csv(out_path, index=False)

print("\n✅ STRICT IFND_multimodel.csv created (text + image only)")
print("Saved at:", out_path)

print("\nFinal shape:", df.shape)


Original shape: (56714, 3)
Columns: ['id', 'Clean_Text', 'Label']

✅ STRICT IFND_multimodel.csv created (text + image only)
Saved at: /content/drive/MyDrive/major_project/IFND_dataset/IFND_multimodel.csv

Final shape: (34527, 4)


In [ ]:
# ==============================================
# STEP 3 — TRUE duplicate removal
# (Clean_Text + ACTUAL IMAGE CONTENT + Label)
# ==============================================
!pip install imagehash

import os
import pandas as pd
from PIL import Image
import imagehash
from tqdm import tqdm

BASE = "/content/drive/MyDrive/major_project/IFND_dataset"

# Load multimodal CSV
df = pd.read_csv(f"{BASE}/IFND_multimodel.csv")

print("Before deduplication:", df.shape)

# Compute perceptual hash for actual image
def compute_phash(rel_path):
    try:
        full_path = os.path.join(BASE, rel_path)
        img = Image.open(full_path).convert("RGB")
        return str(imagehash.phash(img))
    except:
        return None

# Generate image hashes
tqdm.pandas(desc="Reading actual images & computing hashes")
df["image_hash"] = df["image_path"].progress_apply(compute_phash)

# Drop rows where image couldn't be read (very rare)
df = df[df["image_hash"].notna()].reset_index(drop=True)

# 🔥 TRUE duplicate removal
df_dedup = df.drop_duplicates(
    subset=["Clean_Text", "image_hash", "Label"],
    keep="first"
).reset_index(drop=True)

print("After deduplication :", df_dedup.shape)
print("Removed duplicates :", df.shape[0] - df_dedup.shape[0])

# Remove hash column (not needed for training)
df_dedup = df_dedup.drop(columns=["image_hash"])

# Save final cleaned multimodal dataset
out_path = f"{BASE}/finally_multimodel_IFND_is_cleaned.csv"
df_dedup.to_csv(out_path, index=False)

print("\n✅ FINAL multimodal dataset saved")
print("Path:", out_path)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.7 MB/s eta 0:00:00
Before deduplication: (34527, 4)


Reading actual images & computing hashes: 100%|██████████| 34527/34527 [03:00<00:00, 191.46it/s]


After deduplication : (29533, 5)
Removed duplicates : 4994

✅ FINAL multimodal dataset saved
Path: /content/drive/MyDrive/major_project/IFND_dataset/finally_multimodel_IFND_is_cleaned.csv


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

base_path = "/content/drive/MyDrive/major_project/IFND_dataset"

final_df = pd.read_csv(f"{base_path}/finally_multimodel_IFND_is_cleaned.csv")

# Use only required columns
data = final_df[["id", "Clean_Text", "image_path", "Label"]].copy()

# 80% Train, 20% Temp
train_df, temp_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["Label"]
)

# 10% Val, 10% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["Label"]
)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# Save splits
train_df.to_csv(f"{base_path}/IFND_train.csv", index=False)
val_df.to_csv(f"{base_path}/IFND_val.csv", index=False)
test_df.to_csv(f"{base_path}/IFND_test.csv", index=False)

print("✅ Train / Val / Test CSVs saved")


Train: (23626, 4)
Val  : (2953, 4)
Test : (2954, 4)
✅ Train / Val / Test CSVs saved


In [ ]:
for fname in ["IFND_train.csv", "IFND_val.csv", "IFND_test.csv"]:
    d = pd.read_csv(f"{base_path}/{fname}")
    print("\n", fname)
    print(d["Label"].value_counts(normalize=True).round(3))



 IFND_train.csv
Label
REAL    0.679
FAKE    0.321
Name: proportion, dtype: float64

 IFND_val.csv
Label
REAL    0.679
FAKE    0.321
Name: proportion, dtype: float64

 IFND_test.csv
Label
REAL    0.679
FAKE    0.321
Name: proportion, dtype: float64


In [ ]:
# ==============================================
# STEP — Create final_images folder
# (only images actually used in final CSV)
# ==============================================

import os
import shutil
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/major_project/IFND_dataset"

SRC_IMG_DIR = f"{BASE}/images_clean"
DST_IMG_DIR = f"{BASE}/final_images"

os.makedirs(DST_IMG_DIR, exist_ok=True)

# Load final multimodal CSV
df = pd.read_csv(f"{BASE}/finally_multimodel_IFND_is_cleaned.csv")

print("Total rows in final CSV:", len(df))

# Extract image filenames
image_files = df["image_path"].apply(lambda x: os.path.basename(x)).tolist()

copied = 0
missing = 0

for img_name in tqdm(image_files, desc="Copying final images"):
    src_path = os.path.join(SRC_IMG_DIR, img_name)
    dst_path = os.path.join(DST_IMG_DIR, img_name)

    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        copied += 1
    else:
        missing += 1

print("\n✅ FINAL IMAGES SUMMARY")
print("Copied images :", copied)
print("Missing images:", missing)
print("Final images folder:", DST_IMG_DIR)


Total rows in final CSV: 29533


Copying final images: 100%|██████████| 29533/29533 [1:42:20<00:00,  4.81it/s]



✅ FINAL IMAGES SUMMARY
Copied images : 29533
Missing images: 0
Final images folder: /content/drive/MyDrive/major_project/IFND_dataset/final_images


In [ ]:
import os

BASE = "/content/drive/MyDrive/major_project/IFND_dataset"
FINAL_IMG_DIR = f"{BASE}/final_images"

final_images = [f for f in os.listdir(FINAL_IMG_DIR) if f.lower().endswith(".jpg")]

print("🖼️ Total final images:", len(final_images))


🖼️ Total final images: 29533


In [ ]:
import pandas as pd

df = pd.read_csv(f"{BASE}/finally_multimodel_IFND_is_cleaned.csv")

print("CSV rows :", len(df))
print("Images   :", len(final_images))


CSV rows : 29533
Images   : 29533


In [ ]:
import os

BASE = "/content/drive/MyDrive/major_project/IFND_dataset"

folders = {
    "RAW images": f"{BASE}/images",
    "CLEAN images": f"{BASE}/images_clean",
    "FINAL images": f"{BASE}/final_images"
}

def get_folder_stats(folder_path):
    total_size = 0
    total_files = 0

    for root, dirs, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(".jpg"):
                total_files += 1
                total_size += os.path.getsize(os.path.join(root, f))

    size_mb = total_size / (1024 * 1024)
    size_gb = size_mb / 1024
    return total_files, size_mb, size_gb


print("📊 IMAGE FOLDER STATISTICS\n")

for name, path in folders.items():
    files, mb, gb = get_folder_stats(path)
    print(f"{name}")
    print(f"  Images : {files}")
    print(f"  Size   : {mb:.2f} MB  ({gb:.2f} GB)")
    print("-" * 40)


📊 IMAGE FOLDER STATISTICS

RAW images
  Images : 38578
  Size   : 2952.32 MB  (2.88 GB)
----------------------------------------
CLEAN images
  Images : 34527
  Size   : 925.79 MB  (0.90 GB)
----------------------------------------
FINAL images
  Images : 29533
  Size   : 789.22 MB  (0.77 GB)
----------------------------------------


In [ ]:
import os

BASE = "/content/drive/MyDrive/major_project/IFND_dataset"

folders = {
    "RAW images": f"{BASE}/images",
    "CLEAN images": f"{BASE}/images_clean",
    "FINAL images": f"{BASE}/final_images"
}

print("📊 IMAGE COUNT SUMMARY\n")

for name, path in folders.items():
    count = len([f for f in os.listdir(path) if f.lower().endswith(".jpg")])
    print(f"{name}: {count} images")


📊 IMAGE COUNT SUMMARY

RAW images: 38578 images
CLEAN images: 34527 images
FINAL images: 29533 images


In [ ]:
import pandas as pd

BASE = "/content/drive/MyDrive/major_project/IFND_dataset"

csv_path = f"{BASE}/finally_multimodel_IFND_is_cleaned.csv"

df = pd.read_csv(csv_path)

print("Before update:")
print(df["image_path"].head())

# 🔁 replace images_clean → final_images
df["image_path"] = df["image_path"].str.replace(
    "images_clean/",
    "final_images/",
    regex=False
)

print("\nAfter update:")
print(df["image_path"].head())

# overwrite same file (final version)
df.to_csv(csv_path, index=False)

print("\n✅ image_path successfully updated to final_images/")
print("Saved at:", csv_path)


Before update:
0    images_clean/2.jpg
1    images_clean/3.jpg
2    images_clean/4.jpg
3    images_clean/5.jpg
4    images_clean/6.jpg
Name: image_path, dtype: object

After update:
0    final_images/2.jpg
1    final_images/3.jpg
2    final_images/4.jpg
3    final_images/5.jpg
4    final_images/6.jpg
Name: image_path, dtype: object

✅ image_path successfully updated to final_images/
Saved at: /content/drive/MyDrive/major_project/IFND_dataset/finally_multimodel_IFND_is_cleaned.csv


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

base_path = "/content/drive/MyDrive/major_project/IFND_dataset"

final_df = pd.read_csv(f"{base_path}/finally_multimodel_IFND_is_cleaned.csv")

# Use only required columns
data = final_df[["id", "Clean_Text", "image_path", "Label"]].copy()

# 80% Train, 20% Temp
train_df, temp_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["Label"]
)

# 10% Val, 10% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["Label"]
)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# Save splits
train_df.to_csv(f"{base_path}/IFND_train.csv", index=False)
val_df.to_csv(f"{base_path}/IFND_val.csv", index=False)
test_df.to_csv(f"{base_path}/IFND_test.csv", index=False)

print("✅ Train / Val / Test CSVs saved")


Train: (23626, 4)
Val  : (2953, 4)
Test : (2954, 4)
✅ Train / Val / Test CSVs saved


In [ ]:
for fname in ["IFND_train.csv", "IFND_val.csv", "IFND_test.csv"]:
    d = pd.read_csv(f"{base_path}/{fname}")
    print("\n", fname)
    print(d["Label"].value_counts(normalize=True).round(3))



 IFND_train.csv
Label
REAL    0.679
FAKE    0.321
Name: proportion, dtype: float64

 IFND_val.csv
Label
REAL    0.679
FAKE    0.321
Name: proportion, dtype: float64

 IFND_test.csv
Label
REAL    0.679
FAKE    0.321
Name: proportion, dtype: float64
